In [2]:
%pip install -q kagglehub scipy

In [3]:
import kagglehub

path = kagglehub.dataset_download("viacheslavshalamov/russian-road-signs-segmentation-dataset")
print(path)

Using Colab cache for faster access to the 'russian-road-signs-segmentation-dataset' dataset.
/kaggle/input/russian-road-signs-segmentation-dataset


In [1]:
import json
import os
import random
import sys
from pathlib import Path

import cv2
import numpy as np
import torch
from scipy.optimize import linear_sum_assignment
from torch.utils.data import DataLoader, Dataset
from torchvision.models.detection import maskrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

SRC_TRAIN = Path("/kaggle/input/russian-road-signs-segmentation-dataset/sign_dataset/train")
SRC_VAL = Path("/kaggle/input/russian-road-signs-segmentation-dataset/sign_dataset/val")
WORK = Path("/kaggle/working")
STREET_DIR = WORK / "street_signs"
STREET_DIR.mkdir(parents=True, exist_ok=True)

NUM_CLASSES = 9
IMG_SIZE = 320
EPOCHS = 5
BATCH = 4
LR = 0.005
MAX_VAL = 300
SCORE_THR = 0.5
IOU_MATCH = 0.5
IMAGE_EXTS = {".jpg", ".jpeg", ".png"}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
WORKERS = 0 if sys.platform == "darwin" else min(4, (os.cpu_count() or 2) - 1)

In [2]:
def parse_coco_json(json_path: Path) -> tuple[list[np.ndarray], list[int]]:
    data = json.loads(json_path.read_text(encoding="utf-8", errors="ignore"))
    raw_masks = data.get("masks", [])
    class_ids = data.get("class_ids", [])
    masks, labels = [], []
    for i, raw_m in enumerate(raw_masks):
        m = np.array(raw_m, dtype=bool)
        if not m.any():
            continue
        cid = int(class_ids[i]) if i < len(class_ids) else 0
        masks.append(m)
        labels.append(max(1, min(cid + 1, NUM_CLASSES - 1)))
    return masks, labels

In [3]:
def build_cache(src_dir: Path, cache_dir: Path) -> None:
    cache_dir.mkdir(parents=True, exist_ok=True)
    imgs = sorted(src_dir.glob("*.jpg"))
    for img_path in imgs:
        npz_path = cache_dir / f"{img_path.stem}.npz"
        if npz_path.exists():
            continue
        json_path = src_dir / f"{img_path.name}_coco.json"
        raw_masks, labels = parse_coco_json(json_path) if json_path.is_file() else ([], [])
        if raw_masks:
            masks_r = np.stack([
                cv2.resize(m.astype(np.uint8), (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST).astype(bool)
                for m in raw_masks
            ])
        else:
            masks_r = np.zeros((0, IMG_SIZE, IMG_SIZE), dtype=bool)
        np.savez_compressed(str(npz_path), masks=masks_r, labels=np.array(labels, dtype=np.int32))


CACHE_TRAIN = WORK / "cache_train"
CACHE_VAL = WORK / "cache_val"
print("building train cache...")
build_cache(SRC_TRAIN, CACHE_TRAIN)
print("building val cache...")
build_cache(SRC_VAL, CACHE_VAL)
print("done")


class SignDataset(Dataset):
    def __init__(self, src_dir: Path, cache_dir: Path):
        imgs = sorted(src_dir.glob("*.jpg"))
        self.img_paths = [str(p) for p in imgs]
        self.npz_paths = [str(cache_dir / f"{p.stem}.npz") for p in imgs]

    def __len__(self) -> int:
        return len(self.img_paths)

    def __getitem__(self, idx: int):
        img = cv2.cvtColor(cv2.imread(self.img_paths[idx]), cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        image_tensor = torch.from_numpy(img).permute(2, 0, 1).float() / 255.0

        npz = np.load(self.npz_paths[idx])
        masks_np = list(npz["masks"])
        labels_list = npz["labels"].tolist()

        if masks_np:
            masks_t = torch.from_numpy(np.stack(masks_np)).bool()
            labels_t = torch.tensor(labels_list, dtype=torch.int64)
            boxes = []
            for m in masks_np:
                ys, xs = np.where(m)
                boxes.append([xs.min(), ys.min(), xs.max(), ys.max()])
            boxes_t = torch.tensor(boxes, dtype=torch.float32)
        else:
            masks_t = torch.zeros((0, IMG_SIZE, IMG_SIZE), dtype=torch.bool)
            labels_t = torch.zeros(0, dtype=torch.int64)
            boxes_t = torch.zeros((0, 4), dtype=torch.float32)

        target = {
            "masks": masks_t,
            "labels": labels_t,
            "boxes": boxes_t,
            "image_id": torch.tensor([idx]),
        }
        return image_tensor, target


def collate(batch):
    return tuple(zip(*batch))


train_ds = SignDataset(SRC_TRAIN, CACHE_TRAIN)
val_ds = SignDataset(SRC_VAL, CACHE_VAL)
print(f"train={len(train_ds)}, val={len(val_ds)}")
train_dl = DataLoader(train_ds, batch_size=BATCH, shuffle=True, num_workers=0, collate_fn=collate)
val_dl = DataLoader(val_ds, batch_size=1, shuffle=False, num_workers=0, collate_fn=collate)

building train cache...
building val cache...
done
train=2054, val=127


In [4]:
model = maskrcnn_resnet50_fpn(weights="DEFAULT", min_size=IMG_SIZE, max_size=IMG_SIZE)

in_box = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_box, NUM_CLASSES)

in_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels
model.roi_heads.mask_predictor = MaskRCNNPredictor(in_mask, 256, NUM_CLASSES)

model.to(device)
params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=LR, momentum=0.9, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.5)

Downloading: "https://download.pytorch.org/models/maskrcnn_resnet50_fpn_coco-bf2d0c1e.pth" to /root/.cache/torch/hub/checkpoints/maskrcnn_resnet50_fpn_coco-bf2d0c1e.pth


100%|██████████| 170M/170M [00:00<00:00, 183MB/s] 


In [5]:
CKPT = WORK / "maskrcnn_signs.pth"

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss, steps = 0.0, 0
    for imgs, targets in train_dl:
        try:
            imgs = [i.to(device) for i in imgs]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
            loss = sum(model(imgs, targets).values())
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            steps += 1
        except RuntimeError:
            optimizer.zero_grad()
    scheduler.step()
    print(f"epoch {epoch}/{EPOCHS}  loss={total_loss / max(steps, 1):.4f}")

torch.save(model.state_dict(), CKPT)


epoch 1/5  loss=0.8362
epoch 2/5  loss=0.7288
epoch 3/5  loss=0.7018
epoch 4/5  loss=0.6547
epoch 5/5  loss=0.6326


In [6]:
def iou_pair(a: np.ndarray, b: np.ndarray) -> float:
    return float((a & b).sum()) / float((a | b).sum() + 1e-9)


def l2_pair(a: np.ndarray, b: np.ndarray) -> float:
    d = a.astype(np.float64) - b.astype(np.float64)
    return float(np.sqrt(np.mean(d * d) + 1e-12))


@torch.no_grad()
def evaluate(model, dataloader, max_imgs, score_thr, iou_match) -> dict:
    model.eval()
    ious, l2s, img_ious = [], [], []
    tp = fp = fn = n = 0

    for imgs, targets in dataloader:
        if max_imgs and n >= max_imgs:
            break
        preds = model([i.to(device) for i in imgs])

        for pred, tgt in zip(preds, targets):
            n += 1
            keep = pred["scores"] > score_thr
            pr_m = (pred["masks"][keep, 0] > 0.5).cpu().numpy().astype(bool)
            pr_l = pred["labels"][keep].cpu().numpy().tolist()
            gt_m = tgt["masks"].numpy().astype(bool)
            gt_l = tgt["labels"].numpy().tolist()

            if gt_m.shape[0] == 0:
                fp += len(pr_m)
                continue

            mat = np.zeros((len(pr_m), len(gt_m)))
            for i in range(len(pr_m)):
                for j in range(len(gt_m)):
                    if pr_l[i] == gt_l[j]:
                        mat[i, j] = iou_pair(pr_m[i], gt_m[j])

            ri, ci = linear_sum_assignment(1.0 - mat)
            matched_p, matched_g = set(), set()
            local = []
            for i, j in zip(ri, ci):
                iou = float(mat[i, j])
                if iou < 1e-6:
                    continue
                matched_p.add(i)
                matched_g.add(j)
                ious.append(iou)
                local.append(iou)
                l2s.append(l2_pair(pr_m[i], gt_m[j]))
                tp += int(iou >= iou_match)
                if iou < iou_match:
                    fp += 1
                    fn += 1

            fp += len(pr_m) - len(matched_p)
            fn += len(gt_m) - len(matched_g)
            if local:
                img_ious.append(float(np.mean(local)))

    def pct(thr: float) -> float:
        return round(float(np.mean([v >= thr for v in img_ious])), 4) if img_ious else 0.0

    return {
        "mean_iou": round(float(np.mean(ious)) if ious else 0.0, 4),
        "mean_l2": round(float(np.mean(l2s)) if l2s else 0.0, 4),
        "precision": round(tp / (tp + fp + 1e-9), 4),
        "recall": round(tp / (tp + fn + 1e-9), 4),
        "images_evaluated": len(img_ious),
        "pct_iou>=0.50": pct(0.50),
        "pct_iou>=0.75": pct(0.75),
        "pct_iou>=0.90": pct(0.90),
    }

In [7]:
val_metrics = evaluate(model, val_dl, MAX_VAL, SCORE_THR, IOU_MATCH)
print(json.dumps(val_metrics, indent=2, ensure_ascii=False))

{
  "mean_iou": 0.7572,
  "mean_l2": 0.384,
  "precision": 0.7716,
  "recall": 0.0236,
  "images_evaluated": 115,
  "pct_iou>=0.50": 0.9478,
  "pct_iou>=0.75": 0.6435,
  "pct_iou>=0.90": 0.3304
}
